In [1]:
import os
os.environ["SPARK_LOCAL_DIRS"] = "/tmp"               # env var fallback
os.environ["PYSPARK_PYTHON"] = os.environ.get("PYSPARK_PYTHON", "python")  # sometimes needed

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ZoomcampTest") \
    .master("local[*]") \
    .config("spark.local.dir", "/tmp") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.sql.warehouse.dir", "/tmp/spark-warehouse") \
    .config("spark.hadoop.fs.file.impl", "org.apache.hadoop.fs.LocalFileSystem") \
    .config("spark.hadoop.fs.file.impl.disable.cache", "true") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "2g") \
    .config("spark.ui.enabled", "false") \
    .getOrCreate()

print("Spark temporary directory:", spark.conf.get("spark.local.dir"))
print("Spark version:", spark.version)
spark.sparkContext.setLogLevel("WARN")  # less noisy

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/02 16:44:31 WARN Utils: Your hostname, codespaces-b285a2, resolves to a loopback address: 127.0.0.1; using 10.0.3.211 instead (on interface eth0)
26/03/02 16:44:31 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/02 16:44:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/02 16:44:33 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).


Spark temporary directory: /tmp
Spark version: 4.1.1


In [3]:
!wget -O /tmp/taxi_zone_lookup.csv https://github.com/DataTalksClub/nyc-tlc-data/releases/download/misc/taxi_zone_lookup.csv

--2026-03-02 16:45:37--  https://github.com/DataTalksClub/nyc-tlc-data/releases/download/misc/taxi_zone_lookup.csv
Resolving github.com (github.com)... 20.207.73.82
Connecting to github.com (github.com)|20.207.73.82|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/513814948/5a2cc2f5-b4cd-4584-9c62-a6ea97ed0e6a?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-02T17%3A21%3A51Z&rscd=attachment%3B+filename%3Dtaxi_zone_lookup.csv&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-02T16%3A21%3A42Z&ske=2026-03-02T17%3A21%3A51Z&sks=b&skv=2018-11-09&sig=6qM7VhqpxXfF4MK8yNtJ02hhA%2BTXkxpSs53XMR6RbGg%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3MjQ3MDIzNywibmJmIjoxNzcyNDY5OTM3LCJwYXRoIjoicmVsZWFzZWFzc2V0c

In [4]:
!head /tmp/taxi_zone_lookup.csv

"LocationID","Borough","Zone","service_zone"
1,"EWR","Newark Airport","EWR"
2,"Queens","Jamaica Bay","Boro Zone"
3,"Bronx","Allerton/Pelham Gardens","Boro Zone"
4,"Manhattan","Alphabet City","Yellow Zone"
5,"Staten Island","Arden Heights","Boro Zone"
6,"Staten Island","Arrochar/Fort Wadsworth","Boro Zone"
7,"Queens","Astoria","Boro Zone"
8,"Queens","Astoria Park","Boro Zone"
9,"Queens","Auburndale","Boro Zone"


In [5]:
!head -n 5 /tmp/taxi_zone_lookup.csv

"LocationID","Borough","Zone","service_zone"
1,"EWR","Newark Airport","EWR"
2,"Queens","Jamaica Bay","Boro Zone"
3,"Bronx","Allerton/Pelham Gardens","Boro Zone"
4,"Manhattan","Alphabet City","Yellow Zone"


In [6]:
df_zones = spark.read.option("header", "true").csv("/tmp/taxi_zone_lookup.csv")
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [11]:
df_zones.write.parquet('/tmp/zones')

In [13]:
!ls -lh /tmp

total 508K
drwxr-xrw-+  2 codespace codespace 4.0K Mar  2 16:46 artifacts-449d4461-6b4d-4450-8b0a-17aef39f2057
drwxr-xrw-+ 18 codespace codespace 4.0K Mar  2 17:02 blockmgr-99a1c0b3-715e-492b-b9f4-1ad5b24826a2
-rw-r--rw-   1 root      root       12K Mar  2 16:18 dockerd.log
drwxr-xr--+  2 codespace codespace 4.0K Mar  2 16:44 hsperfdata_codespace
-rw-r--rw-   1 codespace codespace 152K Mar  2 16:46 liblz4-java-7131783511231313672.so
-rw-r--rw-   1 codespace codespace    0 Mar  2 16:46 liblz4-java-7131783511231313672.so.lck
drwx------+  2 codespace codespace 4.0K Mar  2 16:23 mcp-Tv4xFJ
drwx------+  2 codespace codespace 4.0K Mar  2 16:18 mcp-oG4UzJ
drwxr-xrw-+  3 codespace codespace 4.0K Mar  2 16:38 node-compile-cache
drwx------+  2 codespace codespace 4.0K Mar  2 16:59 pyright-6210-BQxbMhZ03YQW
drwxr-xrw-+  3 codespace codespace 4.0K Mar  2 16:24 python-languageserver-cancellation
-rwxr--rw-   1 codespace codespace 275K Mar  2 16:49 snappy-1.1.10-97983194-d633-4bcb-83ab-f82af7efc416-